# Font generation

## Vowel and Consonant to line mapping

In [89]:
# Vowels/consonants to glyphs are represented using IPA characters to binary literals
# where the binary literals are little Endian representations of which lines are drawn
# according to the below diagram:
#
#    /|\
#   3 | 4
#  /  |  \
# |\  9  /
# | 8 | 10 
# |  \|/  
# |   |  
# 2   11  
# |   |
# |  /|\
# | 7 | 5
# |/  |  \
#  \  6  /
#   1 | 0
#    \|/
#    (12)

vowels = {
    "ʊɹ"  : 0b0000000011101,
    "ɪɹ"  : 0b0000000001101,
    "ɛɹ"  : 0b0000000000101,
    "ɑːɹ" : 0b0000000011011,
    "oʊ"  : 0b0000000011111,
    "aʊ"  : 0b0000000000001,
    "ɔɪ"  : 0b0000000000010,
    "eɪ"  : 0b0000000001000,
    "aɪ"  : 0b0000000010000,
    "uː"  : 0b0000000011110,
    "ɑː"  : 0b0000000001100,
    "iː"  : 0b0000000001111,
    "ɜː"  : 0b0000000010111,
    "æ"   : 0b0000000011100,
    "ə"   : 0b0000000011000,
    "ɛ"   : 0b0000000000111,
    "ʊ"   : 0b0000000000110,
    "ɪ"   : 0b0000000000011,
    "_"   : 0b0000000000000
}

consonants = {
    "tʃ"  : 0b0100101000000,
    "dʒ"  : 0b0101010000000,
    "ŋ"   : 0b0111111100000,
    "ʒ"   : 0b0111110100000,
    "θ"   : 0b0111101000000,
    "ʃ"   : 0b0110111100000,
    "t"   : 0b0110101000000,
    "w"   : 0b0010100000000,
    "s"   : 0b0111011000000,
    "ɹ"   : 0b0111001000000,
    "k"   : 0b0111000100000,
    "p"   : 0b0110001000000,
    "f"   : 0b0110011000000,
    "ɡ"   : 0b0110001100000,
    "d"   : 0b0101010100000,
    "n"   : 0b0000110100000,
    "m"   : 0b0000010100000,
    "z"   : 0b0101101100000,
    "j"   : 0b0101101000000,
    "v"   : 0b0101100100000,
    "h"   : 0b0101001100000,
    "ð"   : 0b0101011100000,
    "l"   : 0b0101001000000,
    "b"   : 0b0101000100000,
    "_"   : 0b0000000000000
}

# Custom number glyphs (credit to: https://github.com/dirdam for designing these)

numbers = {
    "0": 0b1000000000000,
    "1": 0b0001000000000,
    "2": 0b0010100000000,
    "3": 0b0011100000000,
    "4": 0b0010110100000,
    "5": 0b0011110100000,
    "6": 0b0010110111000,
    "7": 0b0011110111000,
    "8": 0b0010110111011,
    "9": 0b0011110111011
}

## Helper functions for drawing strokes and glyphs

In [90]:
import math

# width = 400
# height = 600
hex_coords = [
    (400,100),
    (200,0),
    (0,100),
    (0,500),
    (200,600),
    (400,500),
    (200,400),
    (200,200)
]

line_map = [
    (0,1), #0
    (1,2), #1
    (2,3), #2
    (3,4), #3
    (4,5), #4
    (0,7), #5
    (1,7), #6
    (2,7), #7
    (3,6), #8
    (4,6), #9
    (5,6), #10
    (7,6) #11
]

THICKNESS = 40
WIDTH = 400
STRIKETHROUGH_HEIGHT = 300
STRIKETHROUGH_SPACING = 80
DIACRITIC_RADIUS = 50

def draw_glyph(pen, glyph_binary: int, strikethrough = False):
    for i in range(12):
        if 1 & (glyph_binary >> i):
            p1 = hex_coords[line_map[i][0]]
            p2 = hex_coords[line_map[i][1]]
            if strikethrough and i in [2,11]:
                p1_end = (p1[0], STRIKETHROUGH_HEIGHT - STRIKETHROUGH_SPACING)
                p2_end = (p1[0], STRIKETHROUGH_HEIGHT)
                draw_line(pen, p1[0],p1[1],p1_end[0],p1_end[1])
                draw_line(pen, p2[0],p2[1],p2_end[0],p2_end[1])
            else:
                draw_line(pen, p1[0], p1[1], p2[0], p2[1])

            draw_circle(pen, p1[0], p1[1], 0, hollow=False)
            draw_circle(pen, p2[0], p2[1], 0, hollow=False)

    if 1 & (glyph_binary >> 12):
        draw_circle(pen, WIDTH/2,-DIACRITIC_RADIUS,DIACRITIC_RADIUS)
    
    if strikethrough:
        draw_line(pen, -THICKNESS/2, STRIKETHROUGH_HEIGHT, WIDTH+THICKNESS/2, STRIKETHROUGH_HEIGHT)


def draw_line(pen, x1, y1, x2, y2):
    """Draw a filled rectangle representing a stroke from (x1,y1) to (x2,y2)."""
    dx = x2 - x1
    dy = y2 - y1
    length = math.hypot(dx, dy)
    # Perpendicular unit vector
    px = -dy / length * (THICKNESS / 2)
    py =  dx / length * (THICKNESS / 2)
    # Four corners of the stroke rectangle
    pen.moveTo((x1 + px, y1 + py))
    pen.lineTo((x2 + px, y2 + py))
    pen.lineTo((x2 - px, y2 - py))
    pen.lineTo((x1 - px, y1 - py))
    pen.closePath()

def draw_circle(pen, cx, cy, radius, hollow = True):
    """Draw a filled outlined of a circle with center (cx,cy) with `radius`"""

    def draw_circle_bezier(radius, clockwise = True):
        a = 1.00005507808
        b = 0.55342925736
        c = 0.99873327689
        r = radius
        f = 1 if clockwise else -1

        pen.moveTo((cx, cy + a*r))
        pen.curveTo(
            (cx + f*b*r, cy + c*r),
            (cx + f*c*r, cy + b*r),
            (cx + f*a*r, cy)
        )
        pen.curveTo(
            (cx + f*c*r, cy - b*r),
            (cx + f*b*r, cy - c*r),
            (cx, cy - a*r)
        )
        pen.curveTo(
            (cx - f*b*r, cy - c*r),
            (cx - f*c*r, cy - b*r),
            (cx - f*a*r, cy)
        )
        pen.curveTo(
            (cx - f*c*r, cy + b*r),
            (cx - f*b*r, cy + c*r),
            (cx, cy + a*r)
        )
        pen.closePath()
    
    outer_radius = radius + THICKNESS/2
    inner_radius = radius - THICKNESS/2
    draw_circle_bezier(outer_radius, True)
    if hollow:
        draw_circle_bezier(inner_radius, False)



## Font generator
Both a struck through font and a regular font can be generated by setting `strikethrough` accordingly

In [91]:
from pathlib import Path
from fontParts.fontshell import RFont
from ufo2ft import compileOTF, compileTTF
import json

def generate_trunic_font(style:str, strikethrough:bool):
    font = RFont()
    
    font_name = f"Trunic-{style}"

    # Basic metadata
    font.info.familyName = "Trunic"
    font.info.styleName = style
    font.info.unitsPerEm = 800
    font.info.ascender = 600
    font.info.descender = -100
    font.info.xHeight = 600
    font.info.capHeight = 600

    font.info.openTypeHheaAscender = 600
    font.info.openTypeHheaDescender = -100
    font.info.openTypeHheaLineGap = 0

    font.info.openTypeOS2TypoAscender = 600
    font.info.openTypeOS2TypoDescender = -100
    font.info.openTypeOS2TypoLineGap = 0

    font.info.openTypeOS2WinAscent = 600
    font.info.openTypeOS2WinDescent = 100

    i = 0
    unicode_mapping = {}
    for inverted in [False, True]:
        for c, c_bin in consonants.items():
            for v, v_bin in vowels.items():

                # Skip blank glyph
                if c == '_' and v == '_':
                    continue

                # Single phonemes are redundant when inverted
                if inverted and (c == '_' or v == '_'):
                    continue

                # Prioritise rhotic vowels over inverted vowel+'ɹ' combinations
                # to prevent collisions
                if inverted and v in ['ʊ','ɪ','ɛ','ɑː'] and c == 'ɹ':
                    continue

                if not inverted:
                    ipa_phoneme = f"{c}{v}"
                else:
                    ipa_phoneme = f"{v}{c}"
                ipa_phoneme = ipa_phoneme.replace('_','')

                g = font.newGlyph(f"trunic_{i:x}")
                g.width = 400
                g.unicode = 0xE000 + i
                
                unicode_mapping[ipa_phoneme] = g.unicode

                pen = g.getPen()

                inverted_bin = 1 << 12 if inverted else 0
                glyph_binary = inverted_bin | c_bin | v_bin
                draw_glyph(pen, glyph_binary, strikethrough)

                i += 1
    
    for n_i, n_bin in numbers.items():
        g = font.newGlyph(f"trunic_number_{n_i}")
        g.width = 400
        g.unicode = 0xE000 + i
        
        unicode_mapping[n_i] = g.unicode

        pen = g.getPen()

        draw_glyph(pen, n_bin, strikethrough)

        i += 1

    font.kerning.clear()

    with open("Data/unicode_mapping.json","w") as f:
        f.write(json.dumps(unicode_mapping, indent=2))

    # Save and compile as ttf
    build_dir = Path("../font/")
    build_dir.mkdir(exist_ok=True)

    ttf = compileTTF(font.naked())
    ttf_path = build_dir / f"{font_name}.ttf"
    ttf.save(str(ttf_path))

    print("Built:")
    print(" ", ttf_path)

generate_trunic_font("Regular", False)
generate_trunic_font("Strikethrough", True)

Built:
  ..\font\Trunic-Regular.ttf
Built:
  ..\font\Trunic-Strikethrough.ttf


## Generate Trunic and display locally 

In [19]:
from trunic_generator import english_to_trunic
from IPython.display import HTML, display
import base64
import os

# WINDOWS ONLY
# Make sure you place the espeak-ng dll file in the src folder for this cell to work
os.environ['PHONEMIZER_ESPEAK_LIBRARY'] = "libespeak-ng.dll"

def load_trunic_font(ttf_path):
    with open(ttf_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f"""
    <style>
      @font-face {{
        font-family: 'Trunic';
        src: url('data:font/truetype;base64,{b64}') format('truetype');
      }}
    </style>
    """))

def show_text(glyph_unicode_list, size=100):
    unicode_text = ''.join([c if not isinstance(c,int) else f"&#x{c:x}" for c in glyph_unicode_list])
    display(HTML(f"""
        <p style="font-family: 'Trunic', serif; font-size: {size}px;">
            {unicode_text}
        </p>
    """))

font_path = r"../font/Trunic-Strikethrough.ttf"
load_trunic_font(font_path)

text = "Hey, what's up guys it's me Charlie"
glyph_unicode_list = english_to_trunic(text)
show_text(glyph_unicode_list,size=40)

raw: heɪ, wʌts ʌp ɡaɪz ɪts miː tʃɑːɹli 
normalized: heɪ, wəts əp ɡaɪz ɪts miː tʃɑːɹliː 


## Generate just HTML unicode text for Trunic font

In [3]:
from src.trunic_generator import english_to_trunic
from IPython.display import HTML, display
import base64
import os

# WINDOWS ONLY
# Make sure you place the espeak-ng dll file in the src folder for this cell to work
os.environ['PHONEMIZER_ESPEAK_LIBRARY'] = "libespeak-ng.dll"

def load_trunic_font(ttf_path):
    with open(ttf_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f"""
    <style>
      @font-face {{
        font-family: 'Trunic';
        src: url('data:font/truetype;base64,{b64}') format('truetype');
      }}
    </style>
    """))

def show_text(glyph_unicode_list, size=100):
    display(HTML(f"""
        <p style="font-family: 'Trunic', serif; font-size: {size}px;">
            {unicode_text}
        </p>
    """))

font_path = r"build\Trunic-Strikethrough.ttf"
load_trunic_font(font_path)

text = "Trunic Generator"
glyph_unicode_list = english_to_trunic(text)
unicode_text = ''.join([c if not isinstance(c,int) else f"&#x{c:x}" for c in glyph_unicode_list])
print(f"HTML unicode text: {unicode_text}")

ModuleNotFoundError: No module named 'src'

In [22]:
from phonemizer import phonemize

# punctuation_pattern = r",.–-(){}[]%!?*\n!\"\'1234567890"
characters_to_preserve = "1234567890,.–-(){}[]<>/\\%!?*^:;`¬|\n!\"°#~+£$&"
text = """hello
world"""

ipa = phonemize(
    text,
    language="en-us",
    backend="espeak",
    preserve_punctuation=True,
    punctuation_marks=characters_to_preserve
)
print(ipa)

həloʊ
wɜːld 


In [12]:
from trunic_generator import english_to_trunic 

text = "beyond"
glyphs = english_to_trunic(text)
show_text(glyphs)

raw: bᵻjɔnd 
normalized: biːjɑːnd 


c:\Users\Ricardo\Documents\Coding\Python\Trunic generator\src\trunic_generator.py:83: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  re_sep = "[\W\n\s]"
c:\Users\Ricardo\Documents\Coding\Python\Trunic generator\src\trunic_generator.py:83: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  re_sep = "[\W\n\s]"


NameError: name 'show_text' is not defined